In [1]:
import os

In [2]:
os.environ['CUDA_LAUNCH_BLOCKING'] = '1'

In [20]:
from copy import deepcopy
import torch
import sys
import torch.nn as nn
import torch.nn.functional as F

from DCLS.construct.modules import Dcls3_1d
from torch.cuda import amp
from spikingjelly.activation_based import functional, surrogate, neuron, layer
from spikingjelly.activation_based.model import parametric_lif_net
from spikingjelly.datasets.dvs128_gesture import DVS128Gesture
from spikingjelly.datasets.dvs_lip import DVSLip
from torch.utils.data import DataLoader
import time
import os
import argparse
import datetime

In [21]:
torch.manual_seed(1)

In [22]:
T = 16
b = 128
j = 8
lr = 0.01
epochs = 100

channels = 16
kernel_size = 3
max_delay = 3

number_of_classes = 11

dataset_class = DVS128Gesture

data_dir = os.path.expanduser('~/datasets/DVSGesture/')

In [23]:
device = 'cuda:0'

In [24]:
class Dcls3_1_SJ(Dcls3_1d):
    def __init__(
        self,
        in_channels,
        out_channels,
        kernel_count,
        learn_delay=True,
        stride=1,
        spatial_padding=0,
        dense_kernel_size=1,
        dilated_kernel_size=1,
        groups=1,
        bias=True,
        padding_mode='zeros',
        version='gauss',
    ):
        super().__init__(
            in_channels,
            out_channels,
            kernel_count,
            stride,
            (*spatial_padding, 0),
            dense_kernel_size,
            dilated_kernel_size,
            groups,
            bias,
            padding_mode,  
            version,
        )
        self.learn_delay = learn_delay
        self.dilated_kernel_size = dilated_kernel_size
        if not self.learn_delay:
            torch.nn.init.constant_(self.P, dilated_kernel_size // 2)
            self.P.requires_grad = False

    def forward(self, x):
        x = x.permute(1, 2, 3, 4, 0) # [T, N, C, H, W] -> [N, C, H, W, T]
        x = F.pad(x, (self.dilated_kernel_size-1, 0), mode='constant', value=0)
        x = super().forward(x)
        x = x.permute(4, 0, 1, 2, 3) # [N, C, H, W, T] -> [T, N, C, H, W]
        return x

In [25]:
class DVSNet(nn.Module):
    def __init__(self,
                 channels=channels,
                 spiking_neuron: callable=None,
                 delayed=False,
                 learn_delay=False,
                 kernel_size=kernel_size,
                 max_delay=max_delay,
                 **kwargs):
        super().__init__()

        conv = []
        ## Stem
        conv.append(layer.Conv2d(2, channels, kernel_size=2, stride=2, 
                                    bias=False))
        conv.append(layer.BatchNorm2d(channels))

        ## Middle Layers
        for i in range(4):
            if delayed:
                version = 'gauss' if learn_delay else 'v1'
                conv.append(
                    Dcls3_1_SJ(in_channels=channels, out_channels=channels, kernel_count=1,
                               learn_delay=learn_delay, dense_kernel_size=kernel_size, dilated_kernel_size=max_delay,
                               bias=False, groups=channels, spatial_padding=(kernel_size//2, kernel_size//2)
                              )        
                )
            else:
                conv.append(layer.Conv2d(channels, channels,
                                         kernel_size=kernel_size, groups=channels,
                                         padding='same', bias=False)
                )
            conv.append(layer.Conv2d(channels, channels, kernel_size=1))
            conv.append(layer.BatchNorm2d(channels))    
            conv.append(spiking_neuron(**deepcopy(kwargs)))
            if i != 3:
                conv.append(layer.Conv2d(channels, 2*channels, kernel_size=2, bias=False, stride=2))
                channels = channels * 2
        

        self.conv = nn.Sequential(
            *conv, 
            layer.AdaptiveAvgPool2d((1, 1))
        )
        self.fc = layer.Linear(in_features=channels, out_features=number_of_classes)
        
    def forward(self, x: torch.Tensor):
        x = self.conv(x).mean((3, 4))
        x = self.fc(x)
            
        return x


In [26]:
train_set = dataset_class(root=data_dir, train=True, data_type='frame', frames_number=T, split_by='number')
test_set = dataset_class(root=data_dir, train=False, data_type='frame', frames_number=T, split_by='number')

The directory [/home/tahaf/datasets/DVSGesture/frames_number_16_split_by_number] already exists.
The directory [/home/tahaf/datasets/DVSGesture/frames_number_16_split_by_number] already exists.


In [27]:
train_data_loader = torch.utils.data.DataLoader(
    dataset=train_set,
    batch_size=b,
    shuffle=True,
    drop_last=True,
    num_workers=j,
    pin_memory=True
)

test_data_loader = torch.utils.data.DataLoader(
    dataset=test_set,
    batch_size=b,
    shuffle=True,
    drop_last=False,
    num_workers=j,
    pin_memory=True
)

In [28]:
scaler = amp.GradScaler()

In [29]:
def check_model(kernel_size, channels, delayed):
    print(f'Results for kernel size = {kernel_size} and delayed = {delayed} and channels = {channels}')
    max_test_acc = -1

    net = DVSNet(
        channels=channels,
        kernel_size=kernel_size,
        spiking_neuron=neuron.LIFNode,
        surrogate_function=surrogate.ATan(),
        delayed=delayed,
        detach_reset=True
    )
    net.to(device)
    num_params = sum(p.numel() for p in net.parameters())
    functional.set_step_mode(net, step_mode='m')

    optimizer = torch.optim.Adam(net.parameters(), lr=lr)
    lr_scheduler = torch.optim.lr_scheduler.OneCycleLR(optimizer, total_steps=round(epochs*60000/b), max_lr=lr)
    
    start_time = time.time()
    for epoch in range(epochs):
        net.train()
        train_loss = 0
        train_acc = 0
        train_samples = 0
        t1 = time.time()
        for frame, label in train_data_loader:
            optimizer.zero_grad()
            frame = frame.to(device)
            frame = frame.transpose(0, 1)  # [N, T, C, H, W] -> [T, N, C, H, W]
            label = label.to(device)
            label_onehot = F.one_hot(label, number_of_classes).float()
    
            if scaler is not None:
                with amp.autocast():
                    out_fr = net(frame)
                    loss = functional.temporal_efficient_training_cross_entropy(out_fr, label)
                scaler.scale(loss).backward()
                scaler.step(optimizer)
                scaler.update()
            else:
                out_fr = net(frame)
                loss = functional.temporal_efficient_training_cross_entropy(out_fr, label)
                loss.backward()
                optimizer.step()
    
            train_samples += label.numel()
            train_loss += loss.item() * label.numel()
            train_acc += (out_fr.mean(0).argmax(1) == label).float().sum().item()
    
            functional.reset_net(net)
    
        train_loss /= train_samples
        train_acc /= train_samples
    
        lr_scheduler.step()
    
        net.eval()
        test_loss = 0
        test_acc = 0
        test_samples = 0
        with torch.no_grad():
            for frame, label in test_data_loader:
                frame = frame.to(device)
                frame = frame.transpose(0, 1)  # [N, T, C, H, W] -> [T, N, C, H, W]
                label = label.to(device)
                label_onehot = F.one_hot(label, number_of_classes).float()
                out_fr = net(frame)
                loss = functional.temporal_efficient_training_cross_entropy(out_fr, label)
                test_samples += label.numel()
                test_loss += loss.item() * label.numel()
                test_acc += (out_fr.mean(0).argmax(1) == label).float().sum().item()
                functional.reset_net(net)
        test_loss /= test_samples
        test_acc /= test_samples
        max_test_acc = max(max_test_acc, test_acc)
        t2 = time.time()
        
        print(f'epoch = {epoch}, train_loss ={train_loss: .4f}, train_acc ={train_acc: .4f}, test_loss ={test_loss: .4f}, test_acc ={test_acc: .4f}, max_test_acc ={max_test_acc: .4f}, time = {t2 - t1: .4f}')
    end_time = time.time()
    total_time = end_time - start_time
    print(f'total time: {total_time}s')
    print('-'*1000)

    return {
        'accuracy': max_test_acc,
        'total_time': total_time,
        'num_params': num_params,
    }

In [13]:
kernel_sizes = [3, 5, 7]
channels = [16, 32]
results = {}

In [14]:
result = check_model(channels=16, kernel_size=7, delayed=False)

Results for kernel size = 7 and delayed = False and channels = 16
epoch = 0, train_loss = 2.3911, train_acc = 0.1076, test_loss = 2.4049, test_acc = 0.0833, max_test_acc = 0.0833, time =  22.5071
epoch = 1, train_loss = 2.3540, train_acc = 0.2118, test_loss = 2.4045, test_acc = 0.0833, max_test_acc = 0.0833, time =  19.6377
epoch = 2, train_loss = 2.3148, train_acc = 0.2934, test_loss = 2.4041, test_acc = 0.0833, max_test_acc = 0.0833, time =  19.6689
epoch = 3, train_loss = 2.2662, train_acc = 0.3568, test_loss = 2.4038, test_acc = 0.0833, max_test_acc = 0.0833, time =  19.4953
epoch = 4, train_loss = 2.2087, train_acc = 0.3993, test_loss = 2.4034, test_acc = 0.0833, max_test_acc = 0.0833, time =  19.6329
epoch = 5, train_loss = 2.1436, train_acc = 0.4323, test_loss = 2.3786, test_acc = 0.1215, max_test_acc = 0.1215, time =  19.7075
epoch = 6, train_loss = 2.0776, train_acc = 0.4505, test_loss = 2.1607, test_acc = 0.4167, max_test_acc = 0.4167, time =  19.5505
epoch = 7, train_loss = 

In [30]:
result2 = check_model(channels=16, kernel_size=7, delayed=True)

Results for kernel size = 7 and delayed = True and channels = 16
epoch = 0, train_loss = 2.3779, train_acc = 0.2222, test_loss = 2.4003, test_acc = 0.0833, max_test_acc = 0.0833, time =  44.8916
epoch = 1, train_loss = 2.3298, train_acc = 0.2839, test_loss = 2.3999, test_acc = 0.0833, max_test_acc = 0.0833, time =  44.7934
epoch = 2, train_loss = 2.2708, train_acc = 0.3090, test_loss = 2.3996, test_acc = 0.0833, max_test_acc = 0.0833, time =  44.9498
epoch = 3, train_loss = 2.2080, train_acc = 0.3203, test_loss = 2.3993, test_acc = 0.0833, max_test_acc = 0.0833, time =  44.9616
epoch = 4, train_loss = 2.1439, train_acc = 0.3620, test_loss = 2.3989, test_acc = 0.0833, max_test_acc = 0.0833, time =  45.2008
epoch = 5, train_loss = 2.0878, train_acc = 0.3906, test_loss = 2.3351, test_acc = 0.2326, max_test_acc = 0.2326, time =  44.8969
epoch = 6, train_loss = 2.0461, train_acc = 0.3967, test_loss = 2.0661, test_acc = 0.4028, max_test_acc = 0.4028, time =  44.8683
epoch = 7, train_loss = 1

In [ ]:
import os
import json

with open(os.path.expanduser('~/delay_results.json'), 'w') as f:
    json.dump(result, f, indent=6) 

In [ ]:
result

In [ ]:
for channel in channels:
    results[channel] = {}
    for kernel_size in kernel_sizes:
        results[channel][kernel_size] = {}
        for is_seperable in [False, True]:
            
            results[channel][kernel_size][is_seperable] = result

In [ ]:
results